# Week 47, Day 4: Brain states: arousal, neuromodulation and the inverted U

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w047/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Where performance peaks

Compute A, sigma and d' = A / sigma on a grid of arousal from 0 to 1, and find where d' peaks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(2015)
a = np.linspace(0, 1, 101)
A = ...
sigma = ...
dp = A / sigma
print(f"peak sensitivity {dp.max():.2f} at arousal {a[np.argmax(dp)]:.2f}; at the extremes {dp[0]:.2f} and {dp[-1]:.2f}")
plt.plot(a, A, label="evoked response"); plt.plot(a, sigma, label="background noise"); plt.plot(a, dp / dp.max(), "k", label="d' (scaled)")
plt.xlabel("arousal (pupil)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_peak():
    "sensitivity peaks at intermediate arousal (between 0.3 and 0.65)"
    assert 0.3 < a[np.argmax(dp)] < 0.65
def test_ends():
    "sensitivity at both extremes is below half the peak"
    assert dp[0] < 0.5 * dp.max() and dp[-1] < 0.5 * dp.max()

test_peak()
test_ends()
print("All checks passed")

### Exercise: Linear against quadratic

Write nll(w, X, y), the negative log-likelihood of a logistic regression, and fit it with pupil alone (plus intercept) and with pupil and pupil squared. Report the likelihood ratio statistic and the pupil size at which the quadratic peaks.

In [ ]:
n = 3000
arousal = r.uniform(0, 1, n)
sig_t = 0.3 + 1.2 * np.exp(-arousal / 0.15); A_t = 1 / (1 + np.exp((arousal - 0.7) / 0.08))
stim = r.random(n) < 0.5
x = stim * A_t + sig_t * r.normal(size=n)
correct = (x > 0.5 * A_t) == stim
pupil = arousal + 0.05 * r.normal(size=n)

def nll(w, X, y):
    ...

X1 = np.c_[np.ones(n), pupil]; X2 = np.c_[X1, pupil ** 2]
f1 = minimize(nll, np.zeros(2), args=(X1, correct)); f2 = minimize(nll, np.zeros(3), args=(X2, correct))
print(f"linear model: slope {f1.x[1]:+.2f}; quadratic model: {np.round(f2.x, 2)}; likelihood ratio {2 * (f1.fun - f2.fun):.1f}; peak at pupil {-f2.x[1] / (2 * f2.x[2]):.2f}")
edges = np.linspace(0, 1, 6); idx = np.digitize(pupil, edges)
plt.plot([(edges[i] + edges[i + 1]) / 2 for i in range(5)], [correct[idx == i + 1].mean() for i in range(5)], "o-"); plt.xlabel("pupil"); plt.ylabel("proportion correct"); plt.show()

In [ ]:
import numpy as np
def test_quadratic():
    "the quadratic model improves the log-likelihood by more than 20 (likelihood ratio above 40)"
    assert 2 * (f1.fun - f2.fun) > 40
def test_shape():
    "the squared term is negative and the fitted peak lies between 0.3 and 0.6"
    b0, b1, b2 = f2.x
    assert b2 < 0 and 0.3 < -b1 / (2 * b2) < 0.6

test_quadratic()
test_shape()
print("All checks passed")